# ☕ CoffeeVision — Multi-Model Classification
### ResNet18 · InceptionResNetV2 · YOLOv11 · YOLOv12
### Baseline (Buruk) → Improved (Baik) Pipeline

In [ ]:
# CELL 1 — Install Library
!pip install roboflow tensorflow keras-cv opencv-python matplotlib scikit-learn seaborn pyyaml ultralytics -q
print('✅ Semua library siap!')


In [ ]:
# CELL 2 — Import Library
import os, shutil, glob, yaml, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from roboflow import Roboflow
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, InceptionResNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('⚠️  Pastikan pakai GPU T4: Runtime → Change runtime type → T4 GPU')


In [ ]:
# Configure your Roboflow API key as an environment variable.
# Google Colab example:
# import os
# from google.colab import userdata
# os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("Set ROBOFLOW_API_KEY in your environment before connecting to Roboflow.")

rf = Roboflow(api_key=API_KEY)
print("Roboflow client configured (credential not displayed).")


In [ ]:
# CELL 4 — Download Dataset (coffee-cherry-ripeness-detection-syszn v2)
import os, shutil, glob, yaml
from roboflow import Roboflow

WS       = 'matthews-workspace-qtko4'
PROJ     = 'coffee-cherry-ripeness-detection-syszn'
VER      = 2
RAW_DIR  = '/content/datasets/raw'
CLS_DIR  = '/content/datasets/cls'
DRIVE    = '/content/drive/MyDrive/CoffeeVision/cls_v2'

def find_images(base):
    imgs = []
    for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
        imgs += glob.glob(os.path.join(base,'**',ext), recursive=True)
    return imgs

def get_names(base):
    for root,dirs,files in os.walk(base):
        for f in files:
            if f.endswith('.yaml') or f.endswith('.yml'):
                try:
                    with open(os.path.join(root,f)) as fp:
                        data = yaml.safe_load(fp)
                    names = data.get('names',[])
                    if isinstance(names,dict): names = list(names.values())
                    if names: return names
                except: pass
    return []

def get_split(path):
    p = path.lower().replace('\\','/')
    if '/train' in p or '/training' in p: return 'train'
    if '/valid' in p or '/val' in p:      return 'valid'
    if '/test'  in p or '/testing' in p:  return 'test'
    return 'train'

def get_label_ki(img_path, base):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    for root,dirs,files in os.walk(base):
        if stem+'.txt' in files:
            lbl = os.path.join(root,stem+'.txt')
            with open(lbl) as fp:
                lines=[l.strip() for l in fp if l.strip()]
                if lines:
                    try: return int(lines[0].split()[0])
                    except: pass
    return 0

# Cek Drive dulu
if os.path.exists(DRIVE) and sum(len(f) for _,_,f in os.walk(DRIVE)) > 50:
    print('✅ Dataset dari Google Drive!')
    if os.path.exists(CLS_DIR): shutil.rmtree(CLS_DIR)
    shutil.copytree(DRIVE, CLS_DIR)
else:
    # Download
    if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
    os.makedirs(RAW_DIR, exist_ok=True)

    print(f'⬇️  Downloading {PROJ} v{VER}...')
    proj = rf.workspace(WS).project(PROJ)
    print(f'   Type: {proj.type}')

    downloaded = False
    for fmt in ['yolov8','yolov11','coco','multiclass']:
        try:
            print(f'   {fmt}...', end=' ', flush=True)
            proj.version(VER).download(fmt, location=RAW_DIR)
            imgs = find_images(RAW_DIR)
            print(f'✅ → {len(imgs)} gambar')
            if len(imgs) > 0:
                downloaded = True
                break
            else:
                print(f'   ⚠️ 0 gambar, coba format lain...')
        except Exception as e:
            print(f'❌ ({str(e)[:50]})')

    if not downloaded:
        print('❌ Download gagal! Cek koneksi dan versi dataset di Roboflow.')
    else:
        # Ambil nama kelas
        names = get_names(RAW_DIR)
        print(f'\n   Kelas dari YAML: {names}')

        if not names:
            # Cek dari subfolder
            for split in ['train','valid','test']:
                sp = os.path.join(RAW_DIR, split)
                if os.path.exists(sp):
                    subs = [k for k in os.listdir(sp) if os.path.isdir(os.path.join(sp,k)) and k not in ['images','labels']]
                    if subs: names = subs; break
            # Cari lebih dalam
            if not names:
                for root,dirs,files in os.walk(RAW_DIR):
                    subs = [d for d in dirs if d not in ['images','labels','train','valid','test']]
                    if subs: names = subs; break

        print(f'   Kelas final: {names}')

        # Konversi ke struktur klasifikasi
        if os.path.exists(CLS_DIR): shutil.rmtree(CLS_DIR)
        os.makedirs(CLS_DIR, exist_ok=True)

        all_imgs = find_images(RAW_DIR)
        print(f'\n   Konversi {len(all_imgs)} gambar ke format klasifikasi...')

        for img_path in all_imgs:
            split = get_split(img_path)
            ki    = get_label_ki(img_path, RAW_DIR)
            kelas = names[ki] if names and ki < len(names) else f'class_{ki}'
            kelas = kelas.replace(' ','_').replace('/','_')
            dst   = os.path.join(CLS_DIR, split, kelas)
            os.makedirs(dst, exist_ok=True)
            shutil.copy2(img_path, os.path.join(dst, os.path.basename(img_path)))

        # Simpan ke Drive
        os.makedirs(os.path.dirname(DRIVE), exist_ok=True)
        if os.path.exists(DRIVE): shutil.rmtree(DRIVE)
        shutil.copytree(CLS_DIR, DRIVE)
        print('💾 Tersimpan di Drive!')

# Ringkasan
print('\n' + '='*50)
print('✅ DATASET SIAP')
for split in ['train','valid','test']:
    sp = os.path.join(CLS_DIR, split)
    if os.path.exists(sp):
        kls = [k for k in os.listdir(sp) if os.path.isdir(os.path.join(sp,k))]
        total = sum(len(f) for _,_,f in os.walk(sp))
        print(f'   {split:6}: {total} gambar | kelas: {sorted(kls)}')


In [ ]:
# CELL 5 — Deteksi Path & Nama Kelas
import os

def cari_path(base, depth=0):
    if depth > 5 or not os.path.exists(base): return None, None, None
    train = os.path.join(base, 'train')
    if os.path.exists(train):
        subs = [k for k in os.listdir(train) if os.path.isdir(os.path.join(train,k))]
        if subs:
            v  = os.path.join(base,'valid')
            te = os.path.join(base,'test')
            return train, v if os.path.exists(v) else None, te if os.path.exists(te) else None
    for sub in sorted(os.listdir(base)):
        sp = os.path.join(base, sub)
        if os.path.isdir(sp):
            r = cari_path(sp, depth+1)
            if r[0]: return r
    return None, None, None

TRAIN_PATH, VALID_PATH, TEST_PATH = cari_path(CLS_DIR)

if TRAIN_PATH:
    NAMA_KELAS   = sorted([k for k in os.listdir(TRAIN_PATH) if os.path.isdir(os.path.join(TRAIN_PATH,k))])
    JUMLAH_KELAS = len(NAMA_KELAS)
    print(f'   Kelas terdeteksi: {NAMA_KELAS}')
    # Verifikasi kelas: harus ada ripe, overripe, semi_ripe, unripe
    EXPECTED = ['overripe', 'ripe', 'semi_ripe', 'unripe']
    missing = [k for k in EXPECTED if k not in NAMA_KELAS]
    if missing:
        print(f'   ⚠️  Kelas tidak ditemukan: {missing}')
        print(f'   Kelas yang ada: {NAMA_KELAS}')
    else:
        print(f'   ✅ Semua kelas ditemukan!')
    YOLO_DATA    = CLS_DIR
    print(f'✅ Dataset siap!')
    print(f'   Kelas ({JUMLAH_KELAS}): {NAMA_KELAS}')
    print(f'   Train : {TRAIN_PATH}')
    print(f'   Valid : {VALID_PATH}')
    print(f'   Test  : {TEST_PATH}')
else:
    print('❌ Path tidak ditemukan!')


In [ ]:
# CELL 6 — GRAFIK EDA: Distribusi Dataset
def hitung(path):
    if not path or not os.path.exists(path): return {}
    kls = sorted([k for k in os.listdir(path) if os.path.isdir(os.path.join(path,k))])
    return {k: len([f for f in os.listdir(os.path.join(path,k)) if f.lower().endswith(('.jpg','.jpeg','.png'))]) for k in kls}

tr = hitung(TRAIN_PATH); va = hitung(VALID_PATH); te = hitung(TEST_PATH)

fig, axes = plt.subplots(1,2, figsize=(14,5))
fig.suptitle('Distribusi Dataset — Coffee Cherry Ripeness', fontsize=13, fontweight='bold')

x = np.arange(len(NAMA_KELAS)); w = 0.25
b1 = axes[0].bar(x-w, [tr.get(k,0) for k in NAMA_KELAS], w, label='Train', color='#2db573')
b2 = axes[0].bar(x,   [va.get(k,0) for k in NAMA_KELAS], w, label='Valid', color='#f5a623')
b3 = axes[0].bar(x+w, [te.get(k,0) for k in NAMA_KELAS], w, label='Test',  color='#e53935')
axes[0].set_xticks(x); axes[0].set_xticklabels(NAMA_KELAS, rotation=25, ha='right', fontsize=9)
axes[0].set_title('Jumlah Gambar per Split', fontweight='bold')
axes[0].set_ylabel('Jumlah Gambar'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
for bars in [b1,b2,b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0: axes[0].text(bar.get_x()+bar.get_width()/2, h+0.5, str(int(h)), ha='center', fontsize=7, fontweight='bold')

palette = sns.color_palette('YlOrBr', len(NAMA_KELAS))
axes[1].pie(tr.values(), labels=tr.keys(), autopct='%1.1f%%', colors=palette, startangle=90)
axes[1].set_title('Proporsi Kelas — Training Set', fontweight='bold')

nilai = list(tr.values())
rasio = max(nilai)/min(nilai) if min(nilai) > 0 else 0
status = f'⚠️ Tidak Seimbang (rasio {rasio:.1f}x)' if rasio > 2 else f'✅ Seimbang (rasio {rasio:.1f}x)'
fig.text(0.5, -0.02, status, ha='center', fontsize=10,
         color='#e53935' if rasio > 2 else '#2db573', fontweight='bold')

plt.tight_layout()
plt.savefig('grafik_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Train={sum(tr.values())} | Valid={sum(va.values())} | Test={sum(te.values())}')


In [ ]:
# CELL 6B — Set Class Names (Verifikasi)
# Kelas resmi dari dataset:
# overripe, ripe, semi_ripe, unripe

# Verifikasi dari folder train
if TRAIN_PATH and os.path.exists(TRAIN_PATH):
    detected = sorted([k for k in os.listdir(TRAIN_PATH)
                       if os.path.isdir(os.path.join(TRAIN_PATH, k))])
    print(f'Kelas terdeteksi dari folder: {detected}')

    EXPECTED = ['overripe', 'ripe', 'semi_ripe', 'unripe']
    if set(detected) == set(EXPECTED):
        NAMA_KELAS   = sorted(EXPECTED)  # ['overripe','ripe','semi_ripe','unripe']
        JUMLAH_KELAS = len(NAMA_KELAS)
        print(f'\n✅ Kelas valid!')
    else:
        NAMA_KELAS   = detected
        JUMLAH_KELAS = len(NAMA_KELAS)
        print(f'\n⚠️  Kelas berbeda dari ekspektasi!')
        print(f'   Ekspektasi : {EXPECTED}')
        print(f'   Ditemukan  : {detected}')

    print(f'\nNAMA_KELAS   = {NAMA_KELAS}')
    print(f'JUMLAH_KELAS = {JUMLAH_KELAS}')
    print()
    for idx, k in enumerate(NAMA_KELAS):
        folder = os.path.join(TRAIN_PATH, k)
        n = len([f for f in os.listdir(folder)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f'  [{idx}] {k}: {n} gambar')


In [ ]:
# CELL 7 — Preprocessing BASELINE (Tanpa Augmentasi)
# ⚠️ Konfigurasi BURUK — sengaja untuk menunjukkan overfitting

BATCH_SIZE = 32
simple_gen = ImageDataGenerator(rescale=1./255)  # ← TANPA augmentasi!

# CNN Custom baseline (128x128)
tr_cnn_b = simple_gen.flow_from_directory(TRAIN_PATH, target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_cnn_b = simple_gen.flow_from_directory(VALID_PATH, target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_cnn_b = simple_gen.flow_from_directory(TEST_PATH,  target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# ResNet18 baseline (224x224)
tr_res18_b = simple_gen.flow_from_directory(TRAIN_PATH, target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_res18_b = simple_gen.flow_from_directory(VALID_PATH, target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_res18_b = simple_gen.flow_from_directory(TEST_PATH,  target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# InceptionResNetV2 baseline (299x299)
tr_irv2_b = simple_gen.flow_from_directory(TRAIN_PATH, target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_irv2_b = simple_gen.flow_from_directory(VALID_PATH, target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_irv2_b = simple_gen.flow_from_directory(TEST_PATH,  target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

NAMA_KELAS   = list(tr_res18_b.class_indices.keys())
JUMLAH_KELAS = tr_res18_b.num_classes
print(f'✅ Baseline generators siap!')
print(f'   Kelas ({JUMLAH_KELAS}): {NAMA_KELAS}')
print(f'   Train: {tr_res18_b.samples} | Valid: {va_res18_b.samples} | Test: {te_res18_b.samples}')


In [ ]:
# CELL 8 — Build & Training BASELINE (Konfigurasi Buruk)
# ⚠️ LR tinggi, tanpa Dropout, tanpa BatchNorm

# ══════════════════════════════════════════════
# CNN BASELINE (dari scratch, tanpa BatchNorm)
# ══════════════════════════════════════════════
def build_cnn_baseline(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation='relu')(inputs)   # tanpa BatchNorm!
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)           # tanpa Dropout!
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, out)
    model.compile(optimizer=Adam(0.01), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════
# ResNet18 BASELINE (dari scratch, tanpa BatchNorm)
# ══════════════════════════════════════════════
def build_resnet18_baseline(input_shape, num_classes):
    def res_block(x, filters, stride=1):
        shortcut = x
        x = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
        x = layers.Activation('relu')(x)                  # tanpa BatchNorm!
        x = layers.Conv2D(filters, 3, padding='same')(x)
        if stride != 1 or shortcut.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, 1, strides=stride)(shortcut)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        return x

    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)
    x = res_block(x, 64);  x = res_block(x, 64)
    x = res_block(x, 128, stride=2); x = res_block(x, 128)
    x = res_block(x, 256, stride=2); x = res_block(x, 256)
    x = res_block(x, 512, stride=2); x = res_block(x, 512)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)           # tanpa Dropout!
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, out)
    model.compile(optimizer=Adam(0.01), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════
# InceptionResNetV2 BASELINE
# ══════════════════════════════════════════════
def build_irv2_baseline(num_classes):
    base = InceptionResNetV2(input_shape=(299,299,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)           # tanpa Dropout!
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=Adam(0.01), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('🔧 Membangun model BASELINE...')
model_cnn_b   = build_cnn_baseline((128,128,3), JUMLAH_KELAS)
model_res18_b = build_resnet18_baseline((224,224,3), JUMLAH_KELAS)
model_irv2_b  = build_irv2_baseline(JUMLAH_KELAS)
print(f'   CNN               params: {model_cnn_b.count_params():,}')
print(f'   ResNet18          params: {model_res18_b.count_params():,}')
print(f'   InceptionResNetV2 params: {model_irv2_b.count_params():,}')

# Training BASELINE — tanpa callbacks, tanpa class weight
print('\n🚀 Training CNN BASELINE (10 epoch)...')
hist_cnn_b = model_cnn_b.fit(tr_cnn_b, epochs=10, validation_data=va_cnn_b, verbose=1)

print('\n🚀 Training ResNet18 BASELINE (10 epoch)...')
hist_res18_b = model_res18_b.fit(tr_res18_b, epochs=10, validation_data=va_res18_b, verbose=1)

print('\n🚀 Training InceptionResNetV2 BASELINE (10 epoch)...')
hist_irv2_b = model_irv2_b.fit(tr_irv2_b, epochs=10, validation_data=va_irv2_b, verbose=1)

print('\n✅ Baseline selesai!')
print(f'   CNN               best val_acc: {max(hist_cnn_b.history["val_accuracy"])*100:.2f}%')
print(f'   ResNet18          best val_acc: {max(hist_res18_b.history["val_accuracy"])*100:.2f}%')
print(f'   InceptionResNetV2 best val_acc: {max(hist_irv2_b.history["val_accuracy"])*100:.2f}%')


In [ ]:
# CELL 9 — Training YOLOv11 & YOLOv12 (Classification)
from ultralytics import YOLO
import os

# ── Path dataset YOLO (data.yaml dari download Roboflow) ──
YOLO_YAML = '/content/Coffee-Cherry-Ripeness-Detection-2/data.yaml'

# Cek apakah file yaml ada
if not os.path.exists(YOLO_YAML):
    # Cari data.yaml di semua folder
    print("⚠️ data.yaml tidak ditemukan di path default, mencari...")
    for root, dirs, files in os.walk('/content'):
        for f in files:
            if f == 'data.yaml':
                YOLO_YAML = os.path.join(root, f)
                print(f"✅ data.yaml ditemukan: {YOLO_YAML}")
                break
        if os.path.exists(YOLO_YAML): break
else:
    print(f"✅ data.yaml ditemukan: {YOLO_YAML}")

# Tampilkan isi data.yaml
import yaml
with open(YOLO_YAML) as f:
    yolo_cfg = yaml.safe_load(f)
print(f"\n📋 Kelas YOLO: {yolo_cfg.get('names', [])}")
print(f"   nc: {yolo_cfg.get('nc', '?')}")

# ── Training YOLOv11 ──
print('\n🚀 Training YOLOv11 Classification...')
model_yolo11 = YOLO('yolo11n-cls.pt')
results_yolo11 = model_yolo11.train(
    data    = YOLO_YAML,
    epochs  = 15,
    imgsz   = 224,
    batch   = 32,
    name    = 'coffee_yolo11',
    patience= 5,
    verbose = False,
    resume  = False
)
print(f'✅ YOLOv11 selesai!')
print(f'   Best model: runs/classify/coffee_yolo11/weights/best.pt')

# ── Training YOLOv12 ──
print('\n🚀 Training YOLOv12 Classification...')
model_yolo12 = YOLO('yolo12n-cls.pt')
results_yolo12 = model_yolo12.train(
    data    = YOLO_YAML,
    epochs  = 15,
    imgsz   = 224,
    batch   = 32,
    name    = 'coffee_yolo12',
    patience= 5,
    verbose = False,
    resume  = False
)
print(f'✅ YOLOv12 selesai!')
print(f'   Best model: runs/classify/coffee_yolo12/weights/best.pt')

# ── Ringkasan ──
print('\n' + '='*50)
print('YOLO Training Selesai!')
print(f'  YOLOv11 best val acc: {results_yolo11.results_dict.get("metrics/accuracy_top1", "?")}')
print(f'  YOLOv12 best val acc: {results_yolo12.results_dict.get("metrics/accuracy_top1", "?")}')


In [ ]:
# CELL 10 — GRAFIK BASELINE: Overfitting di 3 Model
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle('Hasil Training BASELINE — Konfigurasi BURUK\n'
             'LR=0.01 | Tanpa Augmentasi | Tanpa Dropout | Tanpa BatchNorm',
             fontsize=12, fontweight='bold', color='#e53935')

hists_b = [
    ('CNN (Baseline)',            hist_cnn_b,   '#e57373', '#c62828'),
    ('ResNet18 (Baseline)',       hist_res18_b, '#ff8a65', '#e64a19'),
    ('InceptionResNetV2 (Baseline)', hist_irv2_b, '#ffb74d', '#f57f17'),
]

for i, (name, hist, col_tr, col_val) in enumerate(hists_b):
    ax_acc = axes[i][0]; ax_loss = axes[i][1]
    ax_acc.plot(hist.history['accuracy'],     label='Train', color=col_tr,  linewidth=2)
    ax_acc.plot(hist.history['val_accuracy'], label='Valid', color=col_val, linewidth=2, linestyle='--')
    ax_acc.set_title(f'{name} — Accuracy', fontweight='bold', color='#e53935')
    ax_acc.set_ylim([0,1.1]); ax_acc.legend(); ax_acc.grid(alpha=0.3)
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')

    gap = max(hist.history['accuracy']) - max(hist.history['val_accuracy'])
    if gap > 0.05:
        ep = len(hist.history['accuracy'])
        ax_acc.annotate(f'⚠️ Overfitting! Gap={gap:.2f}',
            xy=(ep-1, max(hist.history['accuracy'])),
            xytext=(ep*0.4, max(hist.history['accuracy'])*0.7),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=8, color='red', fontweight='bold')

    ax_loss.plot(hist.history['loss'],     label='Train', color=col_tr,  linewidth=2)
    ax_loss.plot(hist.history['val_loss'], label='Valid', color=col_val, linewidth=2, linestyle='--')
    ax_loss.set_title(f'{name} — Loss', fontweight='bold', color='#e53935')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    for ax in [ax_acc, ax_loss]:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('training_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: training_baseline.png')

# Evaluasi baseline
_, acc_cnn_b   = model_cnn_b.evaluate(te_cnn_b,    verbose=0)
_, acc_res18_b = model_res18_b.evaluate(te_res18_b, verbose=0)
_, acc_irv2_b  = model_irv2_b.evaluate(te_irv2_b,  verbose=0)
val11_b = model_yolo11.val(data=YOLO_YAML, split='test')
val12_b = model_yolo12.val(data=YOLO_YAML, split='test')
acc_yolo11_b = val11_b.top1; acc_yolo12_b = val12_b.top1

print(f'\n📊 Akurasi BASELINE (Test Set):')
print(f'   CNN               : {acc_cnn_b*100:.2f}%  ← rendah karena konfigurasi buruk')
print(f'   ResNet18          : {acc_res18_b*100:.2f}%  ← overfitting')
print(f'   InceptionResNetV2 : {acc_irv2_b*100:.2f}%  ← overfitting')
print(f'   YOLOv11           : {acc_yolo11_b*100:.2f}%')
print(f'   YOLOv12           : {acc_yolo12_b*100:.2f}%')

# Confusion matrix baseline
y_true    = te_res18_b.classes
y_pred_cb = np.argmax(model_cnn_b.predict(te_cnn_b,     verbose=0), axis=1)
y_pred_rb = np.argmax(model_res18_b.predict(te_res18_b, verbose=0), axis=1)
y_pred_ib = np.argmax(model_irv2_b.predict(te_irv2_b,  verbose=0), axis=1)

fig2, axes2 = plt.subplots(1, 3, figsize=(20, 5))
fig2.suptitle('Confusion Matrix BASELINE — Banyak Kesalahan Klasifikasi',
              fontsize=12, fontweight='bold', color='#e53935')
for ax, pred, name, cmap in [
    (axes2[0], y_pred_cb, 'CNN',               'Reds'),
    (axes2[1], y_pred_rb, 'ResNet18',          'Oranges'),
    (axes2[2], y_pred_ib, 'InceptionResNetV2', 'YlOrBr'),
]:
    cm = confusion_matrix(y_true, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=NAMA_KELAS, yticklabels=NAMA_KELAS, ax=ax, linewidths=0.5)
    acc_val = np.sum(pred==y_true)/len(y_true)
    ax.set_title(f'{name}\n⚠️ Acc: {acc_val*100:.2f}%', fontweight='bold')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.savefig('confusion_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: confusion_baseline.png')


In [ ]:
# CELL 11 — Preprocessing IMPROVED (Augmentasi Penuh)
# ✅ Konfigurasi optimal

aug_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.6,1.4],
    shear_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    fill_mode='nearest'
)
normal_gen = ImageDataGenerator(rescale=1./255)

# CNN improved (128x128)
tr_cnn_i = aug_gen.flow_from_directory(TRAIN_PATH, target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_cnn_i = normal_gen.flow_from_directory(VALID_PATH, target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_cnn_i = normal_gen.flow_from_directory(TEST_PATH,  target_size=(128,128), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# ResNet18 improved (224x224)
tr_res18_i = aug_gen.flow_from_directory(TRAIN_PATH, target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_res18_i = normal_gen.flow_from_directory(VALID_PATH, target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_res18_i = normal_gen.flow_from_directory(TEST_PATH,  target_size=(224,224), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# InceptionResNetV2 improved (299x299)
tr_irv2_i = aug_gen.flow_from_directory(TRAIN_PATH, target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
va_irv2_i = normal_gen.flow_from_directory(VALID_PATH, target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
te_irv2_i = normal_gen.flow_from_directory(TEST_PATH,  target_size=(299,299), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# Class weight
labels = tr_res18_i.classes
cw = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(cw))
print('✅ Class Weights:')
for idx, w in class_weights.items(): print(f'   {NAMA_KELAS[idx]}: {w:.3f}')

# Visualisasi augmentasi
batch_img, batch_lbl = next(tr_cnn_i)
plt.figure(figsize=(14, 6))
plt.suptitle('Hasil Augmentasi Data Training (IMPROVED)', fontsize=13, fontweight='bold', color='#2db573')
for i in range(min(12, len(batch_img))):
    plt.subplot(3,4,i+1)
    plt.imshow(batch_img[i]); plt.title(NAMA_KELAS[np.argmax(batch_lbl[i])], fontsize=8); plt.axis('off')
plt.tight_layout()
plt.savefig('augmentasi.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Disimpan: augmentasi.png')


In [ ]:
# CELL 12 — Build Model IMPROVED (BatchNorm + Dropout)

# ══════════════════════════════════════════════
# CNN IMPROVED (BatchNorm + Dropout + GAP)
# ══════════════════════════════════════════════
def build_cnn_improved(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)
    # Block 1
    x = layers.Conv2D(32, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)         # ✅ BatchNorm
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)                # ✅ Dropout
    # Block 2
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)
    # Block 3
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)
    # Block 4
    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)     # ✅ GAP
    # FC
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, out)
    model.compile(optimizer=Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════
# ResNet18 IMPROVED (BatchNorm + Dropout)
# ══════════════════════════════════════════════
def build_resnet18_improved(input_shape, num_classes):
    def res_block(x, filters, stride=1):
        shortcut = x
        x = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
        x = layers.BatchNormalization()(x)     # ✅ BatchNorm
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        if stride != 1 or shortcut.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, 1, strides=stride)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        return x

    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)
    x = res_block(x, 64);  x = res_block(x, 64)
    x = res_block(x, 128, stride=2); x = res_block(x, 128)
    x = res_block(x, 256, stride=2); x = res_block(x, 256)
    x = res_block(x, 512, stride=2); x = res_block(x, 512)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)                # ✅ Dropout
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, out)
    model.compile(optimizer=Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════
# InceptionResNetV2 IMPROVED
# ══════════════════════════════════════════════
def build_irv2_improved(num_classes):
    base = InceptionResNetV2(input_shape=(299,299,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)                # ✅ Dropout
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('🔧 Membangun model IMPROVED...')
model_cnn_i   = build_cnn_improved((128,128,3), JUMLAH_KELAS)
model_res18_i = build_resnet18_improved((224,224,3), JUMLAH_KELAS)
model_irv2_i  = build_irv2_improved(JUMLAH_KELAS)
print(f'   CNN               params: {model_cnn_i.count_params():,}')
print(f'   ResNet18          params: {model_res18_i.count_params():,}')
print(f'   InceptionResNetV2 params: {model_irv2_i.count_params():,}')


In [ ]:
# CELL 13 — Training IMPROVED (Callbacks + Class Weight)
def get_cb(name):
    return [
        EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy', verbose=1),
        ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss', verbose=1, min_lr=1e-7),
        ModelCheckpoint(f'model_{name}_best.h5', save_best_only=True, monitor='val_accuracy', verbose=0)
    ]

print('🚀 Training CNN IMPROVED...')
hist_cnn_i = model_cnn_i.fit(
    tr_cnn_i, epochs=25, validation_data=va_cnn_i,
    class_weight=class_weights, callbacks=get_cb('cnn'), verbose=1)

print('\n🚀 Training ResNet18 IMPROVED...')
hist_res18_i = model_res18_i.fit(
    tr_res18_i, epochs=25, validation_data=va_res18_i,
    class_weight=class_weights, callbacks=get_cb('resnet18'), verbose=1)

print('\n🚀 Training InceptionResNetV2 IMPROVED...')
hist_irv2_i = model_irv2_i.fit(
    tr_irv2_i, epochs=25, validation_data=va_irv2_i,
    class_weight=class_weights, callbacks=get_cb('irv2'), verbose=1)

print('\n✅ IMPROVED training selesai!')
print(f'   CNN               best val_acc: {max(hist_cnn_i.history["val_accuracy"])*100:.2f}%')
print(f'   ResNet18          best val_acc: {max(hist_res18_i.history["val_accuracy"])*100:.2f}%')
print(f'   InceptionResNetV2 best val_acc: {max(hist_irv2_i.history["val_accuracy"])*100:.2f}%')


In [ ]:
# CELL 14 — GRAFIK IMPROVED: Stabil & Konvergen
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle('Hasil Training IMPROVED\n'
             'LR=0.0001 | Augmentasi | Dropout | BatchNorm | Class Weight | EarlyStopping',
             fontsize=12, fontweight='bold', color='#2db573')

hists_i = [
    ('CNN (Improved)',               hist_cnn_i,   '#2db573', '#ff7043'),
    ('ResNet18 (Improved)',          hist_res18_i, '#1565c0', '#42a5f5'),
    ('InceptionResNetV2 (Improved)', hist_irv2_i,  '#6a1b9a', '#ab47bc'),
]

for i, (name, hist, col_tr, col_val) in enumerate(hists_i):
    ax_acc = axes[i][0]; ax_loss = axes[i][1]
    ax_acc.plot(hist.history['accuracy'],     label='Train', color=col_tr,  linewidth=2)
    ax_acc.plot(hist.history['val_accuracy'], label='Valid', color=col_val, linewidth=2, linestyle='--')
    ax_acc.set_title(f'{name} — Accuracy', fontweight='bold', color='#2db573')
    ax_acc.set_ylim([0,1.1]); ax_acc.legend(); ax_acc.grid(alpha=0.3)
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')

    ax_loss.plot(hist.history['loss'],     label='Train', color=col_tr,  linewidth=2)
    ax_loss.plot(hist.history['val_loss'], label='Valid', color=col_val, linewidth=2, linestyle='--')
    ax_loss.set_title(f'{name} — Loss', fontweight='bold', color='#2db573')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    for ax in [ax_acc, ax_loss]:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('training_improved.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: training_improved.png')


In [ ]:
# CELL 15 — Evaluasi & Perbandingan SEMUA Model (5 model)

_, acc_cnn_i   = model_cnn_i.evaluate(te_cnn_i,    verbose=0)
_, acc_res18_i = model_res18_i.evaluate(te_res18_i, verbose=0)
_, acc_irv2_i  = model_irv2_i.evaluate(te_irv2_i,  verbose=0)

model_y11_imp = YOLO('runs/classify/coffee_yolo11/weights/best.pt')
model_y12_imp = YOLO('runs/classify/coffee_yolo12/weights/best.pt')
val11_i = model_y11_imp.val(data=YOLO_YAML, split='test')
val12_i = model_y12_imp.val(data=YOLO_YAML, split='test')
acc_yolo11_i = val11_i.top1; acc_yolo12_i = val12_i.top1

MODEL_NAMES  = ['CNN', 'ResNet18', 'InceptionResNetV2', 'YOLOv11', 'YOLOv12']
ACC_BASELINE = [acc_cnn_b*100, acc_res18_b*100, acc_irv2_b*100, acc_yolo11_b*100, acc_yolo12_b*100]
ACC_IMPROVED = [acc_cnn_i*100, acc_res18_i*100, acc_irv2_i*100, acc_yolo11_i*100, acc_yolo12_i*100]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Perbandingan 5 Model — Baseline vs Improved', fontsize=13, fontweight='bold')

# Bar chart baseline vs improved
x = np.arange(len(MODEL_NAMES)); w = 0.35
b1 = axes[0].bar(x-w/2, ACC_BASELINE, w, label='Baseline', color='#e57373', edgecolor='white')
b2 = axes[0].bar(x+w/2, ACC_IMPROVED, w, label='Improved', color='#2db573', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(MODEL_NAMES, rotation=20, ha='right', fontsize=9)
axes[0].set_title('Test Accuracy — Baseline vs Improved', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_ylim([0,115])
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
for bar in [*b1,*b2]:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+0.5, f'{h:.1f}%', ha='center', fontsize=8, fontweight='bold')

# Delta improvement
deltas = [i-b for b,i in zip(ACC_BASELINE, ACC_IMPROVED)]
colors_d = ['#2db573' if d>=0 else '#e53935' for d in deltas]
bars_d = axes[1].bar(MODEL_NAMES, deltas, color=colors_d, edgecolor='white', width=0.5)
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_title('Peningkatan Akurasi (Improved − Baseline)', fontweight='bold')
axes[1].set_ylabel('Δ Accuracy (%)'); axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=20, labelsize=9)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
for bar, val in zip(bars_d, deltas):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+(0.5 if val>=0 else -1.5),
                 f'+{val:.1f}%' if val>=0 else f'{val:.1f}%',
                 ha='center', fontsize=9, fontweight='bold',
                 color='#2db573' if val>=0 else '#e53935')

# Timeline val_accuracy semua model
colors_imp = ['#2db573','#1565c0','#6a1b9a']
for j, (name, hist) in enumerate([('CNN',hist_cnn_i),('ResNet18',hist_res18_i),('InceptionResNetV2',hist_irv2_i)]):
    axes[2].plot(range(1,len(hist.history['val_accuracy'])+1),
                 [v*100 for v in hist.history['val_accuracy']],
                 color=colors_imp[j], linewidth=2, label=name)
axes[2].set_title('Val Accuracy per Epoch — IMPROVED', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val Accuracy (%)')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: perbandingan_model.png')

print('\n' + '='*60)
print('HASIL EVALUASI SEMUA MODEL — TEST SET')
print('='*60)
print(f'{"Model":<22} {"Baseline":>10} {"Improved":>10} {"Delta":>8}')
print('-'*60)
for name, b, i in zip(MODEL_NAMES, ACC_BASELINE, ACC_IMPROVED):
    print(f'{name:<22} {b:>9.2f}% {i:>9.2f}% {i-b:>+7.2f}%')
best_model = MODEL_NAMES[np.argmax(ACC_IMPROVED)]
best_acc   = max(ACC_IMPROVED)
print(f'\n🏆 Model Terbaik: {best_model} ({best_acc:.2f}%)')


In [ ]:
# CELL 16 — Confusion Matrix & Classification Report IMPROVED

y_true    = te_res18_i.classes
y_pred_ci = np.argmax(model_cnn_i.predict(te_cnn_i,       verbose=0), axis=1)
y_pred_ri = np.argmax(model_res18_i.predict(te_res18_i,   verbose=0), axis=1)
y_pred_ii = np.argmax(model_irv2_i.predict(te_irv2_i,     verbose=0), axis=1)

for name, pred in [('CNN', y_pred_ci), ('ResNet18', y_pred_ri), ('InceptionResNetV2', y_pred_ii)]:
    print(f'\n===== {name} IMPROVED =====')
    print(classification_report(y_true, pred, target_names=NAMA_KELAS))

# Confusion matrix 3 model Keras
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Confusion Matrix IMPROVED', fontsize=12, fontweight='bold', color='#2db573')
for ax, pred, name, cmap in [
    (axes[0], y_pred_ci, 'CNN',               'Greens'),
    (axes[1], y_pred_ri, 'ResNet18',          'Blues'),
    (axes[2], y_pred_ii, 'InceptionResNetV2', 'Purples'),
]:
    cm  = confusion_matrix(y_true, pred)
    acc = np.sum(pred==y_true)/len(y_true)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=NAMA_KELAS, yticklabels=NAMA_KELAS, ax=ax,
                linewidths=0.5, annot_kws={'size':9})
    ax.set_title(f'{name}\n✅ Acc: {acc*100:.2f}%', fontweight='bold')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.savefig('confusion_improved.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: confusion_improved.png')

# Precision/Recall/F1 per kelas
fig2, axes2 = plt.subplots(1, 3, figsize=(20, 5))
fig2.suptitle('Precision / Recall / F1-Score per Kelas — IMPROVED', fontsize=12, fontweight='bold')
for ax, pred, name, cmap in [
    (axes2[0], y_pred_ci, 'CNN',               'Greens'),
    (axes2[1], y_pred_ri, 'ResNet18',          'Blues'),
    (axes2[2], y_pred_ii, 'InceptionResNetV2', 'Purples'),
]:
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, labels=range(len(NAMA_KELAS)))
    xk = np.arange(len(NAMA_KELAS)); wk = 0.25
    cm_p = plt.get_cmap(cmap)
    ax.bar(xk-wk, p,  wk, label='Precision', color=cm_p(0.4))
    ax.bar(xk,    r,  wk, label='Recall',    color=cm_p(0.6))
    ax.bar(xk+wk, f1, wk, label='F1-Score',  color=cm_p(0.8))
    ax.set_xticks(xk); ax.set_xticklabels(NAMA_KELAS, rotation=30, ha='right', fontsize=8)
    ax.set_ylim([0,1.2]); ax.set_title(f'{name}', fontweight='bold')
    ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)
    ax.axhline(np.mean(f1), color='red', linestyle='--', linewidth=1.5)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('metrics_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Disimpan: metrics_per_kelas.png')


In [ ]:
# CELL 16B — Grafik Precision, Recall & F1 per Kelas untuk SETIAP Model
# ✅ Satu grafik per model, menampilkan P/R/F1 untuk 4 kelas
# ✅ BUKAN grafik epoch/training

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score, precision_score, recall_score, f1_score
)

# ══════════════════════════════════════════════════════
# STEP 1 — Prediksi dari 3 model Keras
# ══════════════════════════════════════════════════════
print("🔍 Memprediksi dengan semua model...")

y_true       = te_res18_i.classes
y_pred_cnn   = np.argmax(model_cnn_i.predict(te_cnn_i,     verbose=0), axis=1)
y_pred_res18 = np.argmax(model_res18_i.predict(te_res18_i, verbose=0), axis=1)
y_pred_irv2  = np.argmax(model_irv2_i.predict(te_irv2_i,   verbose=0), axis=1)
print(f"✅ Keras selesai! Kelas: {NAMA_KELAS}")

# ══════════════════════════════════════════════════════
# STEP 2 — Prediksi YOLOv11 & YOLOv12 (dengan class mapping)
# ══════════════════════════════════════════════════════
from ultralytics import YOLO

model_y11_f = YOLO('runs/classify/coffee_yolo11/weights/best.pt')
model_y12_f = YOLO('runs/classify/coffee_yolo12/weights/best.pt')

print(f"YOLOv11 class order: {model_y11_f.names}")
print(f"YOLOv12 class order: {model_y12_f.names}")

def collect_test_items(test_path, nama_kelas):
    items = []
    for idx, kelas in enumerate(nama_kelas):
        kelas_path = os.path.join(test_path, kelas)
        if not os.path.isdir(kelas_path): continue
        imgs = [f for f in os.listdir(kelas_path)
                if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for img in sorted(imgs):
            items.append((os.path.join(kelas_path, img), idx))
    return items

def predict_yolo_remap(model, items, nama_kelas, batch=32):
    # Remap YOLO index → global index berdasarkan nama kelas
    remap = {}
    for yolo_idx, yolo_name in model.names.items():
        for g_idx, g_name in enumerate(nama_kelas):
            if g_name.strip().lower() == yolo_name.strip().lower():
                remap[yolo_idx] = g_idx
                break
    paths  = [p for p, _ in items]
    preds  = []
    for i in range(0, len(paths), batch):
        results = model.predict(paths[i:i+batch], verbose=False)
        for r in results:
            top1 = int(r.probs.top1)
            preds.append(remap.get(top1, top1))
    return np.array(preds)

test_items    = collect_test_items(TEST_PATH, NAMA_KELAS)
y_true_yolo   = np.array([idx for _, idx in test_items])
y_pred_yolo11 = predict_yolo_remap(model_y11_f, test_items, NAMA_KELAS)
y_pred_yolo12 = predict_yolo_remap(model_y12_f, test_items, NAMA_KELAS)
print("✅ YOLO selesai!")

# ══════════════════════════════════════════════════════
# STEP 3 — Kumpulkan semua model
# ══════════════════════════════════════════════════════
MODELS = {
    'CNN':               (y_true,      y_pred_cnn),
    'ResNet18':          (y_true,      y_pred_res18),
    'InceptionResNetV2': (y_true,      y_pred_irv2),
    'YOLOv11':           (y_true_yolo, y_pred_yolo11),
    'YOLOv12':           (y_true_yolo, y_pred_yolo12),
}

# Warna per kelas
CLASS_COLORS = {
    'overripe':  '#9b59b6',
    'ripe':      '#e74c3c',
    'semi_ripe': '#f39c12',
    'unripe':    '#2ecc71',
}

MODEL_COLORS = {
    'CNN':               '#2ecc71',
    'ResNet18':          '#3498db',
    'InceptionResNetV2': '#9b59b6',
    'YOLOv11':           '#e67e22',
    'YOLOv12':           '#e74c3c',
}

# ══════════════════════════════════════════════════════
# STEP 4 — Print Classification Report (teks)
# ══════════════════════════════════════════════════════
metrics_summary = {}
for model_name, (yt, yp) in MODELS.items():
    print(f"{'─'*60}")
    print(f"📊 {model_name} — Classification Report")
    print(f"{'─'*60}")
    print(classification_report(yt, yp, target_names=NAMA_KELAS,
                                 labels=range(len(NAMA_KELAS)), zero_division=0))
    metrics_summary[model_name] = {
        'acc': accuracy_score(yt, yp),
        'p':   precision_score(yt, yp, average='weighted', zero_division=0),
        'r':   recall_score(yt, yp,    average='weighted', zero_division=0),
        'f1':  f1_score(yt, yp,        average='weighted', zero_division=0),
    }

best = max(metrics_summary, key=lambda k: metrics_summary[k]['f1'])
print(f"\n🏆 Model terbaik: {best} — F1={metrics_summary[best]['f1']*100:.2f}%")

# ══════════════════════════════════════════════════════
# STEP 5 — SATU GRAFIK PER MODEL: Precision / Recall / F1 per kelas
# ══════════════════════════════════════════════════════
print("\n📊 Membuat grafik per model...")

for model_name, (yt, yp) in MODELS.items():
    p_cls, r_cls, f1_cls, sup = precision_recall_fscore_support(
        yt, yp, labels=range(len(NAMA_KELAS)), zero_division=0)
    acc = accuracy_score(yt, yp)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor('#f8f9fa')
    main_color = MODEL_COLORS[model_name]

    fig.suptitle(
        f'{model_name}  |  Accuracy: {acc*100:.2f}%',
        fontsize=15, fontweight='bold', color=main_color, y=1.02
    )

    metrics_data = [
        ('Precision', p_cls,  '#3498db'),
        ('Recall',    r_cls,  '#e67e22'),
        ('F1-Score',  f1_cls, main_color),
    ]

    for ax, (metric_name, values, bar_color) in zip(axes, metrics_data):
        ax.set_facecolor('#ffffff')

        # Bar per kelas
        x = np.arange(len(NAMA_KELAS))
        bars = ax.bar(
            x, values, width=0.55,
            color=[CLASS_COLORS.get(k, bar_color) for k in NAMA_KELAS],
            edgecolor='white', linewidth=1.2, zorder=3
        )

        # Value labels di atas bar
        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.012,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold',
                color='#2c3e50'
            )

        # Garis rata-rata
        mean_val = np.mean(values)
        ax.axhline(mean_val, color='#e74c3c', linestyle='--', linewidth=1.5,
                   label=f'Mean = {mean_val:.3f}', zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels(NAMA_KELAS, rotation=20, ha='right', fontsize=10)
        ax.set_ylim([0, 1.15])
        ax.set_ylabel(metric_name, fontsize=11, fontweight='bold')
        ax.set_title(metric_name, fontsize=12, fontweight='bold', pad=10)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(axis='y', alpha=0.3, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_facecolor('#fafafa')

    # Legend kelas (warna)
    legend_patches = [
        mpatches.Patch(color=CLASS_COLORS.get(k, '#888'), label=k)
        for k in NAMA_KELAS
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.08),
               fancybox=True, shadow=True)

    plt.tight_layout()
    fname = f'metrics_{model_name.lower().replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
    plt.show()
    print(f"✅ Disimpan: {fname}")

# ══════════════════════════════════════════════════════
# STEP 6 — GRAFIK RINGKASAN: Semua model dalam 1 grafik
# ══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Perbandingan Semua Model — Precision · Recall · F1 per Kelas',
             fontsize=14, fontweight='bold', y=1.02)

model_names = list(MODELS.keys())
n_models    = len(model_names)
n_classes   = len(NAMA_KELAS)
x           = np.arange(n_classes)
w           = 0.15

for ax_idx, metric_name in enumerate(['Precision', 'Recall', 'F1-Score']):
    ax = axes[ax_idx]
    ax.set_facecolor('#fafafa')

    for m_idx, model_name in enumerate(model_names):
        yt, yp = MODELS[model_name]
        p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(
            yt, yp, labels=range(n_classes), zero_division=0)

        vals = {'Precision': p_cls, 'Recall': r_cls, 'F1-Score': f1_cls}[metric_name]
        offset = (m_idx - n_models/2 + 0.5) * w

        bars = ax.bar(x + offset, vals, w,
                      label=model_name,
                      color=MODEL_COLORS[model_name],
                      edgecolor='white', linewidth=0.8, zorder=3)

        for bar, val in zip(bars, vals):
            if val > 0.05:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.01,
                        f'{val:.2f}',
                        ha='center', va='bottom', fontsize=6, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(NAMA_KELAS, rotation=20, ha='right', fontsize=10)
    ax.set_ylim([0, 1.18])
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=10)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax_idx == 0:
        ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.savefig('metrics_semua_model_perband.png', dpi=150,
            bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print("✅ Disimpan: metrics_semua_model_perband.png")

# ══════════════════════════════════════════════════════
# STEP 7 — Tabel ringkasan akhir
# ══════════════════════════════════════════════════════
print("\n" + "="*70)
print("RINGKASAN AKHIR — SEMUA MODEL (Weighted Average)")
print("="*70)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("─"*70)
for m, v in metrics_summary.items():
    mark = " 🏆" if m == best else ""
    print(f"{m:<22} {v['acc']*100:>8.2f}%"
          f" {v['p']*100:>9.2f}%"
          f" {v['r']*100:>7.2f}%"
          f" {v['f1']*100:>7.2f}%{mark}")
print("="*70)
print("\n📁 File tersimpan:")
for model_name in MODELS:
    fname = f'metrics_{model_name.lower().replace(" ","_")}.png'
    print(f"   {fname}")
print("   metrics_semua_model_perband.png")


In [ ]:
# CELL 17 — Simpan & Download Semua Model & Grafik
import shutil
from google.colab import files

model_cnn_i.save('model_cnn_final.h5')
model_res18_i.save('model_resnet18_final.h5')
model_irv2_i.save('model_inceptionresnetv2_final.h5')
print('✅ Model Keras disimpan!')

semua = [
    'model_cnn_final.h5',
    'model_resnet18_final.h5',
    'model_inceptionresnetv2_final.h5',
    'grafik_distribusi.png',
    'augmentasi.png',
    'training_baseline.png',
    'confusion_baseline.png',
    'training_improved.png',
    'perbandingan_model.png',
    'confusion_improved.png',
    'metrics_per_kelas.png',
]
print('\n⬇️  Download file...')
for f in semua:
    try: files.download(f); print(f'  ✅ {f}')
    except: print(f'  ⚠️  {f} tidak ada')

shutil.make_archive('yolo_weights', 'zip', 'runs/classify')
files.download('yolo_weights.zip')
print('  ✅ yolo_weights.zip (YOLOv11 + YOLOv12)')
print('\n✅ Selesai! Total 5 model tersimpan.')
